In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

In [2]:
train = pd.read_csv(r"C:\Users\gupta\Downloads\train.csv")
test = pd.read_csv(r"C:\Users\gupta\Downloads\test.csv")

test_ids = test["PassengerId"]
df = pd.concat([train, test], axis=0).reset_index(drop=True)

In [3]:
df[['Deck','CabinNum','Side']] = df['Cabin'].str.split('/', expand=True)
df.drop(columns='Cabin', inplace=True)

df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')
df['CabinNumBin'] = pd.qcut(df['CabinNum'], 8, duplicates='drop')

In [4]:
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

# Infer CryoSleep if no spending
zero_spend = (df[spend_cols].sum(axis=1) == 0)
df.loc[zero_spend & df['CryoSleep'].isna(), 'CryoSleep'] = True

# CryoSleep passengers cannot spend
df.loc[df['CryoSleep'] == True, spend_cols] = 0

# Total spending + flag
df['TotalSpend'] = df[spend_cols].sum(axis=1)
df['NoSpend'] = (df[spend_cols].sum(axis=1) == 0).astype(int)

In [5]:
df['Group'] = df['PassengerId'].str.split('_').str[0]
df['GroupSize'] = df.groupby('Group')['PassengerId'].transform('count')
df['IsAlone'] = (df['GroupSize'] == 1).astype(int)

In [6]:
df['Age'].fillna(df['Age'].median(), inplace=True)

df['AgeBin'] = pd.cut(
    df['Age'],
    bins=[0,12,18,25,35,50,80],
    labels=False
)

df['AgeBin'].fillna(df['AgeBin'].mode()[0], inplace=True)

C:\Users\gupta\AppData\Local\Temp\ipykernel_9664\2927973553.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\gupta\AppData\Local\Temp\ipykernel_9664\2927973553.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

In [7]:
df['VIP'].fillna(False, inplace=True)
df['CryoSleep'].fillna(False, inplace=True)

for col in ['HomePlanet','Destination','Deck','Side','CabinNumBin']:
    df[col].fillna(df[col].mode()[0], inplace=True)

for col in spend_cols + ['TotalSpend']:
    df[col].fillna(0, inplace=True)

C:\Users\gupta\AppData\Local\Temp\ipykernel_9664\2062411359.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['VIP'].fillna(False, inplace=True)
C:\Users\gupta\AppData\Local\Temp\ipykernel_9664\2062411359.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['VIP'].fillna(False, inplace=True)
C:\Users\gup

In [8]:
for col in spend_cols + ['TotalSpend']:
    df[col] = np.log1p(df[col])

In [9]:
cat_cols = [
    'HomePlanet','CryoSleep','Destination','VIP',
    'Deck','Side','CabinNumBin','AgeBin'
]

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [10]:
df.drop(columns=['PassengerId','Name','Group'], inplace=True)

X = df.iloc[:len(train)].drop(columns='Transported')
y = train['Transported'].astype(int)
X_test = df.iloc[len(train):].drop(columns='Transported')

In [11]:
model = CatBoostClassifier(
    iterations=1600,
    depth=8,
    learning_rate=0.018,
    l2_leaf_reg=6,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=0
)

model.fit(X, y)

In [12]:
probs = model.predict_proba(X_test)[:, 1]

thresholds = [0.46, 0.47, 0.48, 0.49]

for t in thresholds:
    preds = probs > t

    submission = pd.DataFrame({
        "PassengerId": test_ids,
        "Transported": preds.astype(bool)
    })

    filename = f"submission_thresh_{t:.2f}.csv"
    submission.to_csv(filename, index=False)

    print(f"Saved {filename}")

Saved submission_thresh_0.46.csv
Saved submission_thresh_0.47.csv
Saved submission_thresh_0.48.csv
Saved submission_thresh_0.49.csv
